# Option B - swap the VLM family (fast check)

Qwen3-VL isn't locked in. This benchmarks a different **VLM family** on the L40, using the *same* OCR-grounding from Option A so the comparison is apples-to-apples - any difference you see is the model family, not the grounding.

The filter for a candidate is three-way, and family support varies, so **confirm each, don't assume**:
1. **Fits 48 GB** with headroom for image tokens.
2. **Doc/table strength.**
3. **Serves on vLLM *and* trains a hot-loadable LoRA under Unsloth** (your pipeline needs all three).

| Candidate | L40 fit | Doc/table strength | Notes |
|---|---|---|---|
| **PaliGemma 2** (10B/28B) | 10B easy, 28B tight | Table-proven: 97.6% S-TEDS fine-tuned on PubTabNet | But that's *fine-tuned*; zero-shot on invoices is a different regime. |
| **InternVL3** (14B/38B) | 14B comfortable | Among the strongest open doc/OCR VLMs | Best raw doc-understanding candidate. |
| **Gemma 3** (12B/27B) | 12B easy | General, decent OCR | Well-supported Unsloth+vLLM fallback if PaliGemma serving is fiddly. |
| **Qwen2.5-VL-32B** (AWQ/FP8) | Tight | 81.7 TEDS zero-shot (repo) | Same family, bigger; cheapest "stronger model" datapoint. |

Shortlist to try first: **PaliGemma 2** (table-proven) and **InternVL3** (strongest general doc VLM).

## Serving reference (run in a terminal on the L40, one at a time)
New port each so nothing collides with the teacher on 8000.
```bash
# InternVL3-14B
vllm serve OpenGVLab/InternVL3-14B --served-model-name internvl3-14b \
  --max-model-len 8192 --limit-mm-per-prompt image=1 --port 8002

# PaliGemma 2 (10B)
vllm serve google/paligemma2-10b-pt-448 --served-model-name paligemma2-10b \
  --max-model-len 8192 --limit-mm-per-prompt image=1 --port 8003
```
Then uncomment the matching entry in `ENDPOINTS` below. (Exact HF repo ids and any trust-remote-code flags vary by model - check the model card.)

## Config

In [ ]:
import sys, base64, mimetypes
from pathlib import Path
from openai import OpenAI   # pip install openai

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# One entry per served VLM. Serve ONE at a time on the L40 (48 GB) and fill in
# its endpoint, or run several ports if they fit together. 'name' is just a label;
# 'model' must match that server's --served-model-name.
ENDPOINTS = [
    {'name': 'qwen (current)', 'base_url': 'http://localhost:8000/v1', 'model': 'qwen3.6-35b-a3b-fp8'},
    # {'name': 'internvl3-14b',  'base_url': 'http://localhost:8002/v1', 'model': 'internvl3-14b'},
    # {'name': 'paligemma2-10b', 'base_url': 'http://localhost:8003/v1', 'model': 'paligemma2-10b'},
    # {'name': 'gemma3-12b',     'base_url': 'http://localhost:8004/v1', 'model': 'gemma3-12b'},
]

IMAGES_DIR = ROOT / 'data' / 'invoices'
IMAGE_GLOB = ('*.png', '*.jpg', '*.jpeg', '*.webp', '*.tif', '*.tiff')
OCR_STYLE, ROW_TOL, COL_GAP = 'grid', 0.6, 1.0
TEMPERATURE, MAX_TOKENS, REQUEST_TIMEOUT = 0.0, 4096, 300

images = sorted(p for pat in IMAGE_GLOB for p in IMAGES_DIR.glob(pat))
print(len(images), 'invoices;', len(ENDPOINTS), 'endpoints configured')


In [ ]:
from src.ocr.engine import run_ocr
from src.ocr.layout import serialize_layout
from src.model.prompts import GROUNDED_INSTRUCTION, format_layout_block, clean_prediction
from src.data.html_utils import extract_cells

# One OpenAI client per endpoint (base_url is fixed at construction).
CLIENTS = {ep['name']: OpenAI(base_url=ep['base_url'], api_key='EMPTY') for ep in ENDPOINTS}


def data_url(path):
    mime = mimetypes.guess_type(str(path))[0] or 'image/png'
    return f'data:{mime};base64,' + base64.b64encode(Path(path).read_bytes()).decode()


def ask(ep, image_path, instruction, ocr_layout=None):
    text = instruction if ocr_layout is None else f'{instruction}\n\n{format_layout_block(ocr_layout)}'
    resp = CLIENTS[ep['name']].chat.completions.create(
        model=ep['model'], temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
        timeout=REQUEST_TIMEOUT,
        messages=[{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': data_url(image_path)}},
            {'type': 'text', 'text': text}]}],
    )
    return clean_prediction(resp.choices[0].message.content)


## Compare families on one invoice, grounding ON
OCR runs once; each configured endpoint reconstructs from the identical image + layout.

In [ ]:
from IPython.display import HTML, Image as IPyImage, display

assert images, f'no invoices in {IMAGES_DIR}'
img = images[0]

# OCR once; every model gets the identical image + layout, so any difference in
# output is the *family*, not the grounding.
words = run_ocr(img)
layout = serialize_layout(words, style=OCR_STYLE, row_tol=ROW_TOL, col_gap=COL_GAP)

display(IPyImage(filename=str(img), width=420))
rows = []
for ep in ENDPOINTS:
    try:
        html = ask(ep, img, GROUNDED_INSTRUCTION, ocr_layout=layout)
    except Exception as e:
        print(ep['name'], 'FAILED', e); continue
    cells = extract_cells(html)
    rows.append((ep['name'], len(cells), sum(c.is_spanning for c in cells),
                 html.startswith('<table')))
    print('===', ep['name'], '==='); display(HTML(html or '<i>empty</i>'))

print('\nname               cells span  ok')
for n, c, s, ok in rows:
    print(f'{n:18s} {c:5d} {s:4d}  {ok}')


---
**How to read this.** With no ground truth yet, this is a *qualitative* read plus structural counts (cell / span totals) - enough to spot a family that's clearly better or clearly broken, not to rank two close ones. The real ranking needs the **20-invoice eval set** scored with TEDS-Struct through the existing harness (`src/eval/`, `compare_runs`) - which prints `NOT SIGNIFICANT` when a 2-3 point gap on ~20 tables is just noise.

**Before committing to any winner**, verify it also (a) trains a LoRA under Unsloth and (b) hot-loads that adapter into vLLM - that's the pipeline requirement, and it's the step most likely to disqualify an otherwise strong family. A/B one new family at a time so you can attribute the change.